# settings

> One table for each value that you can change, and the file that keeps your changes.

Each value that a person can change is a row in one table, `SETTINGS`. A row has a group, a name, a label, one line of help, a type, a default and, for a number, a range.
The engine, the web app and the Settings page all read this table. Thus each value has one place.

Your changes go in `settings.json` in `cfg_dir()`. The code reads a value when it uses it, so a change works at once, with no restart.
A saved value outside its range gets the nearest end of the range. A saved value of the wrong type gets the default. A bad file never stops the app.

In [ ]:
#| default_exp settings

In [ ]:
#| hide
import os, tempfile
from fastcore.test import *
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

This module imports only the standard library and fastcore. Each engine module reads its values from here, so the import must stay light.

In [ ]:
#| export
import os, json, threading
from pathlib import Path
from fastcore.utils import *

## The groups

The Settings page shows the groups in this order. Each group has a label and one lead line.

In [ ]:
#| export
GROUPS = dict(look=('Look', 'Pick the colours of the app.'),
              learning=('Learning', 'Set how the tutor teaches you. The Profile page shows the same values.'),
              walks=('Walks', 'Pick how long a walk is and how much of it is hands-on.'),
              talks=('Talks', 'Set how a talk runs and how the room works.'),
              checks=('Checks and grading', 'Set how many tries you get and how the tutor grades an answer.'),
              review=('Review', 'Set when an idea comes back for review.'),
              links=('Links', 'Set how much of a link the tutor reads.'),
              runner=('Code runner', 'Set the limits for the code that you run in a check.'),
              stt=('Speech to text', 'Pick the model that writes down what you say.'),
              models=('Models', 'Pick the model for each task. "Default" uses the default model.'),
              lists=('Lists', 'Set how many items show in the lists of the app.'),
              prompts=('Prompts', 'These texts tell the model how to ask, grade, hint and teach.'))

## The table

`add` puts one row in `SETTINGS`. The name is `section.key`: the section of `settings.json` and the key in it. The type comes from the default.
`more` puts the row under the closed "More" part of its group. `store='profile'` marks a value that the vault keeps in your profile, not this file.
`hi` can be the name of another row. Then that value is the top of the range. `ui=False` marks a value that its group shows with its own controls. `reset=False` keeps a value when you use the defaults of its group: the theme, and the speech engine and model.

In [ ]:
#| export
SETTINGS = {}   # 'section.key' -> the row

def add(name:str, group:str, label:str, help:str, dflt, lo=None, hi=None, more:bool=False, choices=None, unit:str='',
        pct:bool=False, store:str='file', ui:bool=True, reset:bool=True):
    "Add the setting `name` to `SETTINGS` and return its row."
    kind = ('bool' if isinstance(dflt, bool) else 'choice' if choices else 'int' if isinstance(dflt, int) else
            'float' if isinstance(dflt, float) else 'str')
    sec, key = name.split('.', 1)
    SETTINGS[name] = r = AttrDict(name=name, section=sec, key=key, group=group, label=label, help=help, kind=kind, dflt=dflt,
                                  lo=lo, hi=hi, more=more, choices=tuple(choices or ()), unit=unit, pct=pct, store=store, ui=ui, reset=reset)
    return r

The **Look** group keeps its own controls: the theme swatches and the mode. The **Learning** group holds the three values of your profile.

In [ ]:
#| export
add('look.theme', 'look', 'Theme', 'The colours of the app.', 'paper', ui=False, reset=False)
add('look.mode', 'look', 'Light or dark', 'Auto follows the Mac.', 'auto', ui=False, reset=False)
add('learning.pace', 'learning', 'Pace', 'How big each step is. Gentle uses small steps. Fast uses bigger steps.', 'steady',
    choices=('gentle', 'steady', 'fast'), store='profile')
add('learning.session_len', 'learning', 'Questions in a session', 'How many questions the tutor asks in one session.', 12, 3, 40, store='profile')
add('learning.explore', 'learning', 'How often to try a new way', 'How often the tutor teaches in a way that you did not try yet.', 'sometimes',
    choices=('never', 'sometimes', 'often'), store='profile');

The **Walks** group sets the size of a walk and of a side walk.

In [ ]:
#| export
add('walks.steps', 'walks', 'Ideas in a walk', 'The most ideas in one walk. With more ideas, the walk is longer. The tutor keeps the other ideas in your concepts.', 12, 3, 30)
add('walks.hands', 'walks', 'Hands-on share', 'How many checks in a walk are a demo, an order or code. The others are short answers.', 1/3, 0., 1., pct=True)
add('walks.chain', 'walks', 'Ideas in a side walk', 'The most ideas in a side walk beside the paper.', 4, 2, 8)
add('walks.passage', 'walks', 'Passage length', 'The most characters of the source that show for each idea.', 600, 200, 2000, more=True, unit='characters')
add('walks.note', 'walks', 'Note length', 'The most words in the tutor note under the passage.', 60, 20, 120, more=True, unit='words')
add('walks.depth', 'walks', 'Rabbit-hole depth', 'How many side walks you can open one inside the other. A deeper idea opens as its own session.', 2, 1, 4, more=True)
add('walks.fresh', 'walks', 'Time for "Back on the paper"', 'How long the "Back on the paper" line shows after you come back from a side walk.', 10, 1, 60, more=True, unit='min')
add('walks.demo_wait', 'walks', 'Wait for a demo', 'How long a step waits for its demo. Then the check is a short answer.', 90, 30, 300, more=True, unit='s');

The **Talks** group sets how a talk runs: the questions, the time, the vote and the room for phones.

In [ ]:
#| export
add('talks.choice', 'talks', 'Choice questions from short checks', 'A new talk turns each short check into a choice question with 4 options. When this is off, the question asks for a short answer.', True)
add('talks.ask_min', 'talks', 'Minutes for each question', 'The minutes that a question adds to its slide in the time of a talk.', 1, 0, 5, unit='min')
add('talks.pair_min', 'talks', 'Time to talk to a neighbour', 'How long the room talks in pairs before a second vote.', 2, 1, 10, unit='min')
add('talks.weak', 'talks', 'Unsure below', 'A slide is unsure when the share of right answers is below this value. The recap lists the unsure slides.', .7, .3, .95, more=True, pct=True)
add('talks.split_lo', 'talks', 'A split vote starts at', 'A vote with a share of right answers from this value to the end value is split. Then the room talks in pairs.', .3, 0., 1., more=True, pct=True)
add('talks.split_hi', 'talks', 'A split vote ends at', 'The top of a split vote. Above it, most of the room knows the answer.', .7, 0., 1., more=True, pct=True)
add('talks.laser_ms', 'talks', 'Laser fade', 'How long the laser dot stays on the audience screen after the pointer stops.', 1500, 500, 5000, more=True, unit='ms')
add('talks.image_mb', 'talks', 'Largest figure for the room', 'A figure larger than this does not go to the room.', 5, 1, 20, more=True, unit='MB')
add('talks.port', 'talks', 'Room port', 'The port of the room that phones join.', 5180, 1024, 65000, more=True)
add('talks.room_hours', 'talks', 'Room closes after', 'The room closes after this time, also when the talk did not end.', 3, 1, 8, more=True, unit='hours')
add('talks.phones', 'talks', 'Most phones', 'The most phones that can join one room.', 300, 10, 300, more=True)
# talks.nets only adds a warning at a new place. It never changes what a server binds to or accepts.
add('talks.nets', 'talks', 'Places you opened a room', 'The Wi-Fi places where you opened a room before. The app warns you at a new place.', '', ui=False);

The **Checks and grading** group sets the tries, the grade of a shaky pass, and when an idea counts as learned.

In [ ]:
#| export
add('checks.tries', 'checks', 'Tries', 'How many answers you can give before the tutor teaches the idea.', 3, 1, 5)
add('checks.sources', 'checks', 'Use my sources', 'The tutor uses what you read and wrote to make its questions.', True)
add('checks.one_ask', 'checks', 'One ask per question', 'When a question asks two things, the tutor writes it again with one ask.', True)
add('checks.near', 'checks', 'Shaky pass', 'A grade at or above this number is solid. A pass below it is shaky, and it comes back sooner.', 7, 5, 9, more=True)
add('checks.goal', 'checks', 'Learned depth', 'The depth at which an idea counts as learned. 2 is recall, 3 is explain, 4 is apply and 5 is transfer.', 4, 2, 5, more=True)
add('checks.high', 'checks', 'Sure confidence', 'At or above this confidence, you count as sure. A sure answer that is wrong counts as a misconception.', 60, 40, 90, more=True, unit='%')
add('checks.hints', 'checks', 'Hints before a pass is shaky', 'With this many hints, a right answer counts as shaky.', 2, 1, 4, more=True)
add('checks.demo_misses', 'checks', 'Misses before a demo', 'After this many misses on an idea, the tutor makes a demo for it.', 2, 1, 5, more=True)
add('checks.demo_ladder', 'checks', 'More help after a demo', 'When you miss an idea after its demo, the tutor teaches it in another way. After the next miss, it shows a simpler demo.', True)
add('checks.retest', 'checks', 'Retest a miss after', 'The tutor asks a missed idea again after this many other questions in the session.', 2, 1, 6, more=True, unit='questions')
add('checks.feedback_row', 'checks', 'Feedback under a demo or a lesson', 'Show a short row under each demo and each lesson: Too simple, Just right or Not clear. The tutor uses what you choose for the next demo or lesson.', True);

The **Review** group sets the time to the next review. A daily limit on review is new: 0 means no limit, as before.

In [ ]:
#| export
add('review.solid', 'review', 'First review after a solid answer', 'When a new idea comes back after your first solid answer.', 1., .25, 7., unit='days')
add('review.per_day', 'review', 'Most reviews a day', 'The tutor stops review after this many questions in a day. 0 means no limit.', 0, 0, 200)
add('review.shaky', 'review', 'First review after a shaky answer', 'When a new idea comes back after your first shaky answer.', .5, .1, 3., more=True, unit='days')
add('review.growth', 'review', 'Growth factor', 'How fast the time to the next review grows after a solid answer. With a higher number, reviews come less often.', 3., 1., 6., more=True)
add('review.fading', 'review', 'Fading below recall', 'An idea is fading when the chance that you recall it is below this value.', .6, .3, .9, more=True, pct=True);

The **Links** group sets how much of a link the tutor gets. The pages read by default can not be more than the page cap.

In [ ]:
#| export
add('links.pages', 'links', 'Pages read by default', 'How many pages of a link the tutor reads to make a walk.', 8, 1, 'links.cap')
add('links.cap', 'links', 'Page cap', 'The most pages that the tutor reads from one link.', 20, 1, 50, more=True)
add('links.timeout', 'links', 'Time for each fetch', 'How long the tutor waits for one page. Then it stops.', 30, 5, 120, more=True, unit='s')
add('links.speed', 'links', 'Download speed cap', 'The fastest speed at which abhyasa gets a page.', 1., .1, 10., more=True, unit='MB/s')
add('links.kept', 'links', 'Characters kept for each page', 'The most characters of one page that the vault keeps.', 400_000, 10_000, 1_000_000, more=True)
add('links.read', 'links', 'Characters read for each page', 'The most characters of one page that the tutor reads to find ideas.', 100_000, 10_000, 1_000_000, more=True);

The **Code runner** group sets the time limits of a run. The kill time and the CPU limit follow the total time: total + 2 s and total + 1 s.

In [ ]:
#| export
add('runner.total', 'runner', 'Total time', 'How long your code can run for all the tests together. Then the run stops.', 10, 5, 60, unit='s')
add('runner.per_test', 'runner', 'Time for each test', 'How long one test can run. Then the test fails.', 2, 1, 10, more=True, unit='s')
add('runner.file_mb', 'runner', 'Largest file', 'The largest file that your code can write.', 10, 1, 100, more=True, unit='MB');

The **Speech to text** group keeps the engine and the model, and the pauses at which the mic cuts what you say.

In [ ]:
#| export
add('stt.engine', 'stt', 'Engine', 'The speech engine.', 'gemma', ui=False, reset=False)
add('stt.model', 'stt', 'Model', 'The speech model. Empty uses the default of the engine.', '', ui=False, reset=False)
add('stt.segment', 'stt', 'Cut at pauses', 'The text shows after each pause. When this is off, the text comes in fixed blocks of time.', True)
add('stt.pause', 'stt', 'Pause before a cut', 'How long a pause must be before the text of what you said shows.', 700, 300, 2000, more=True, unit='ms')
add('stt.longest', 'stt', 'Longest segment', 'The text shows after this time, also when you do not pause.', 12, 5, 30, more=True, unit='s');

The **Models** group has the default model and one model for each task. An empty value uses the default model.
The tasks after `extract` borrow the model of an older task when they have none: `BORROWS` says which. The reply caps are in tokens.

In [ ]:
#| export
TASK_HELP = dict(question='Writes each question and its hidden answer.', grade='Reads your answer and grades it.', hint='Gives the next hint.',
                 lesson='Teaches the smallest step after a miss.', extract='Finds the ideas in a text.', plan='Plans the steps toward a goal.',
                 pick='Picks the main ideas of a long source for a walk.', note='Writes the note under the passage of a step.',
                 profile='Learns about you from your answers.', exercise='Writes a code exercise and its tests.',
                 talk='Writes a choice question for a talk from a short check.')
BORROWS = dict(plan='extract', pick='extract', note='hint', profile='extract', exercise='question', talk='question')
TASK_LABEL = dict(question='Questions', grade='Grades', hint='Hints', lesson='Lessons', extract='Ideas in a text', plan='Plans',
                  pick='Main ideas', note='Walk notes', profile='Your profile', exercise='Code exercises', talk='Talk questions')

add('models.default', 'models', 'Default model', 'The model for each task that has no model of its own. '
    '"abhyasa --model" and $ABHYASA_MODEL come first when they are set.', '', ui=False)
for t, h in TASK_HELP.items():
    add(f'models.{t}', 'models', TASK_LABEL[t], h + (f' Empty uses the model for {TASK_LABEL[BORROWS[t]].lower()}.' if t in BORROWS else ''),
        '', more=t not in ('question', 'grade', 'hint', 'lesson'), ui=False)
for t, n in dict(question=4096, grade=2048, hint=512, lesson=2048, extract=4096).items():
    add(f'tokens.{t}', 'models', f'Reply cap: {TASK_LABEL[t].lower()}', 'The most tokens in one reply. Claude Code models do not use this limit.',
        n, 1, 32768, more=True, unit='tokens')

The **Lists** group sets how many items show in the lists of the app. All of it is under "More".

In [ ]:
#| export
add('lists.strengths', 'lists', 'Weak and strong ideas', 'How many weak spots and strong ideas the Strengths page shows.', 10, 3, 50, more=True)
add('lists.journeys', 'lists', 'Journeys on Today', 'How many of your journeys show on Today.', 3, 1, 10, more=True)
add('lists.beside', 'lists', 'Other journeys on a journey page', 'How many other journeys show at the foot of a journey.', 4, 0, 20, more=True)
add('lists.weeks', 'lists', 'Weeks in your history', 'How many weeks the history of your learning shows.', 8, 1, 52, more=True)
add('lists.search', 'lists', 'Search results', 'How many results from what you learned the search shows.', 12, 3, 50, more=True)
add('lists.elsewhere', 'lists', 'Results from other sources', 'How many results from your other sources the search shows.', 2, 0, 10, more=True)
add('lists.snippet', 'lists', 'Length of a search result', 'The most characters of each search result.', 300, 100, 2000, more=True, unit='characters')
add('lists.attempts', 'lists', 'Answers on a concept page', 'How many of your recent answers a concept page shows.', 10, 3, 50, more=True)
add('lists.changes', 'lists', 'Changes on the Profile page', 'How many changes to your profile the Profile page shows.', 40, 10, 200, more=True)
add('lists.sort_batch', 'lists', 'Concepts in one sort', 'How many concepts the tutor sorts into groups in one request. With fewer, each request is shorter.', 40, 10, 100, more=True);

Each group except **Prompts** has rows, and each row is in a group. The Prompts group lists the prompts of `prompts.PROMPTS`.

In [ ]:
assert set(GROUPS) - {'prompts'} == {r.group for r in SETTINGS.values()}
assert all(r.kind != 'int' or r.lo <= r.dflt for r in SETTINGS.values())
len(SETTINGS)

## The file

`DFLT_SETTINGS` is the default of each section: the rows that the file keeps, the saved prompt texts and the models that the Settings page downloaded.
`settings()` gives the saved settings over the defaults. A section that is not a dict in the file is ignored, and a top-level key that is not known is dropped.

In [ ]:
#| export
def cfg_dir() -> Path:
    "The folder for abhyasa's settings and keys: `$ABHYASA_CONFIG`, else `~/.config/abhyasa`."
    return Path(os.getenv('ABHYASA_CONFIG') or '~/.config/abhyasa').expanduser()

DFLT_SETTINGS = dict(prompts={}, downloaded=[])   # your text for a prompt, by name; the models that the Settings page downloaded
for _r in SETTINGS.values():
    if _r.store == 'file': DFLT_SETTINGS.setdefault(_r.section, {})[_r.key] = _r.dflt

`_saved` reads the file only when it changed, so a value read in a loop costs one `stat`.

In [ ]:
#| export
_cache = {}

def _path() -> Path: return cfg_dir()/'settings.json'

def _saved() -> dict:
    "The saved file as a dict; empty when it is missing or bad. It is read again only when it changed."
    p = _path()
    try: st = p.stat()
    except OSError: return {}
    k = (str(p), st.st_ino, st.st_mtime_ns, st.st_size)
    if _cache.get('k') != k:
        try: d = json.loads(p.read_text())
        except (OSError, ValueError): d = {}
        _cache.update(k=k, d=d if isinstance(d, dict) else {})
    return _cache['d']

def settings() -> dict:
    "The saved settings over the defaults."
    d = json.loads(json.dumps(DFLT_SETTINGS))
    for k, v in _saved().items():
        if isinstance(d.get(k), dict):
            if isinstance(v, dict): d[k].update(v)
        elif k in d and type(v) is type(d[k]): d[k] = v
    return json.loads(json.dumps(d))

`save_settings` writes the file: first a temporary file with its own name, then a rename, so a crash never leaves half a file. A lock keeps two changes from the web app apart. It does not keep a value that is the default. `update_settings` changes keys in one section.

In [ ]:
#| export
_lock = threading.RLock()   # the web app saves from many threads: one read-change-write at a time

def save_settings(d:dict) -> dict:
    "Write all the settings."
    with _lock: return _write(d)

def _write(d:dict) -> dict:
    p = _path()
    p.parent.mkdir(parents=True, exist_ok=True)
    lean = {k: dict(v) if isinstance(v, dict) else v for k, v in d.items()}
    for r in SETTINGS.values():   # a default is not kept, so a new default in a later version works
        sec = lean.get(r.section)
        if isinstance(sec, dict) and r.key in sec and type(sec[r.key]) is type(r.dflt) and sec[r.key] == r.dflt: del sec[r.key]
    tmp = p.with_suffix(f'.{os.getpid()}.{threading.get_ident()}.tmp')   # its own name, so two writes never share a file
    tmp.write_text(json.dumps(lean, indent=2)); tmp.replace(p)
    _cache.clear()
    return d

def update_settings(section:str, **kw) -> dict:
    "Change some keys in one section of the settings."
    with _lock:
        d = settings(); d.setdefault(section, {}).update(kw)
        return save_settings(d)

With no file, `settings()` gives the defaults. A change in one section keeps the others.

In [ ]:
test_eq(settings()['walks']['steps'], 12)
update_settings('stt', segment=False)
test_eq(settings()['stt']['segment'], False)
test_eq(settings()['look'], dict(theme='paper', mode='auto'))
update_settings('stt', segment=True)

A file from before this table still loads: its sections update the defaults, and the new sections get their defaults.

In [ ]:
_path().write_text(json.dumps(dict(stt=dict(engine='mlx', model='m', segment=True), models=dict(question='', grade='', hint='', lesson=''),
                                   tokens=dict(question='', grade='', hint='', lesson='', extract=''), look=dict(theme='ink', mode='dark'), prompts={}, downloaded=[])))
d = settings()
test_eq((d['stt']['engine'], d['stt']['pause'], d['look']['theme'], d['walks']['steps'], d['models']['extract']), ('mlx', 700, 'ink', 12, ''))
save_settings(dict(prompts={}, downloaded=[]))

A file that is not JSON, or a section that is not a dict, gives the defaults. An unknown top-level key is dropped.

In [ ]:
_path().write_text('{not json')
test_eq(settings()['walks']['steps'], 12)
_path().write_text(json.dumps(dict(walks=5, mine=1)))
d = settings()
test_eq((d['walks']['steps'], 'mine' in d), (12, False))
save_settings(dict(prompts={}, downloaded=[]))

## Read one value

`fit` makes a saved value fit its row. A number outside the range gets the nearest end. A value of the wrong type gets the default.
A whole number can come as `12.0` from a form; it stays a whole number. A choice must be one of the choices.

In [ ]:
#| export
def top(r):
    "The top of the range of row `r`: a number, or the value of the row that `r.hi` names."
    return val(r.hi) if isinstance(r.hi, str) else r.hi

def fit(r, v):
    "The value `v` made to fit row `r`: clamped to the range, or the default for the wrong type."
    if r.kind == 'bool': return v if isinstance(v, bool) else r.dflt
    if r.kind == 'choice': return v if v in r.choices else r.dflt
    if r.kind == 'str': return v if isinstance(v, str) else r.dflt
    if isinstance(v, str):   # a number that you typed in the file by hand, such as "8192"
        try: v = float(v.strip())
        except ValueError: return r.dflt
    if isinstance(v, bool) or not isinstance(v, (int, float)) or v != v: return r.dflt
    v = min(max(v, r.lo), top(r))
    return int(round(v)) if r.kind == 'int' else float(v)

`val` gives the value of a row now: the saved value made to fit, else the default. The engine calls it each time it needs the value.

In [ ]:
#| export
def raw(name:str):
    "The saved value of `name` as it is in the file, or None."
    r = SETTINGS[name]
    sec = _saved().get(r.section)
    return sec.get(r.key) if isinstance(sec, dict) else None

def val(name:str):
    "The value of the setting `name` now."
    r = SETTINGS[name]
    if r.store != 'file': raise KeyError(f'{name} is kept in the profile; read it from `Learning.profile()`')
    v = raw(name)
    return r.dflt if v is None else fit(r, v)

def dflt(name:str): return SETTINGS[name].dflt

A saved 50 ideas becomes 30, the most. A string gives the default. A share above 1 becomes 1.

In [ ]:
update_settings('walks', steps=50, hands='a lot', note=40.0)
test_eq((val('walks.steps'), val('walks.hands'), val('walks.note')), (30, 1/3, 40))
update_settings('walks', hands=1.5, steps=1)
test_eq((val('walks.hands'), val('walks.steps')), (1., 3))
save_settings(dict(prompts={}, downloaded=[]))

The top of the pages read by default is the page cap.

In [ ]:
update_settings('links', pages=40, cap=10)
test_eq((val('links.pages'), val('links.cap')), (10, 10))
save_settings(dict(prompts={}, downloaded=[]))

## Change one value

`set_val` saves a value made to fit its row, and returns it. `reset` puts back the default of one row, and `reset_group` of each row in a group.
`changed` is true when a value is not its default. `clamped` gives the value in the file when it was outside the range, so the page can say so.

In [ ]:
#| export
def set_val(name:str, v):
    "Save the value `v` of the setting `name`, made to fit its row. Return the value in use."
    r = SETTINGS[name]
    update_settings(r.section, **{r.key: fit(r, v)})
    return val(name)

def reset(name:str):
    "Use the default of the setting `name` again."
    r = SETTINGS[name]
    if r.store != 'file': raise KeyError(f'{name} is kept in the profile; change it with `Learning.set_profile()`')
    with _lock:
        d = settings(); d[r.section].pop(r.key, None)
        save_settings(d)
    return r.dflt

def rows(group:str) -> list:
    "The rows of `group`, in table order."
    return [r for r in SETTINGS.values() if r.group == group]

def reset_group(group:str) -> list:
    "Use the default of each row of `group` that the file keeps and that a group reset can change. Return their names."
    ns = [r.name for r in rows(group) if r.store == 'file' and r.reset and raw(r.name) is not None]
    with _lock:
        d = settings()
        for n in ns: d[SETTINGS[n].section].pop(SETTINGS[n].key, None)
        save_settings(d)
    return ns

def changed(name:str) -> bool: return val(name) != dflt(name)

def clamped(name:str):
    "The number in the file for `name` when it was outside the range, else None."
    r, v = SETTINGS[name], raw(name)
    if r.kind not in ('int', 'float') or isinstance(v, bool) or not isinstance(v, (int, float)): return None
    return v if v < r.lo or v > top(r) else None

A value saved out of range is saved as the end of the range. `reset` and `reset_group` put back the defaults.

In [ ]:
test_eq(set_val('walks.steps', 99), 30)
test_eq(raw('walks.steps'), 30)
assert changed('walks.steps') and not changed('walks.note')
reset('walks.steps')
test_eq(val('walks.steps'), 12)
set_val('checks.tries', 5); set_val('checks.near', 8)
test_eq(sorted(reset_group('checks')), ['checks.near', 'checks.tries'])
test_eq((val('checks.tries'), val('checks.near')), (3, 7))
update_settings('walks', steps=50)
test_eq(clamped('walks.steps'), 50)
# a number typed by hand as text works, and text that is not a number gives the default
update_settings('tokens', hint='900', grade='many')
test_eq((val('tokens.hint'), val('tokens.grade')), (900, 2048))
# a value that the profile keeps has no reset here
test_fail(lambda: reset('learning.pace'), contains='profile')
save_settings(dict(prompts={}, downloaded=[]))

## What stays fixed

Some values keep you safe, so they have no setting: the closed process of the code runner (no network, the sandbox profile, the audit hook, `python -I`, `sandbox=True`),
the link checks for local addresses, plain HTTP, the link kinds, the HTML allowlist, the check of a pane address, the local-only hosts, the test allowlist, the grade scale and the depth ladder.
The room for phones keeps its safety limits fixed too: the bind rule, the join code, the rate limits, the body size and the headers.
`FIXED` names them, and no row may have one of these words in its name.

In [ ]:
#| export
FIXED = ('network', 'sandbox', 'audit', 'isolate', 'local', 'plain', 'link_kind', 'html', 'pane_url', 'hosts', 'allowlist', 'safe_names', 'grade_scale', 'depths',
         'bind', 'join_code', 'rate', 'body', 'room_headers')

In [ ]:
assert not [n for n in SETTINGS for w in FIXED if w in n]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()